In [2]:
import didppy as dp
import vrplib
import numpy as np
import math
import tempfile
import os

# **Data**

## Solomon C101 dataset

## Reading Data

In [6]:
# Create a temporary file to store the C101_txt content
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_file:
    temp_file.write(C101_txt)
    temp_file_path = temp_file.name

try:
    solomon_data = vrplib.read_instance(temp_file_path, instance_format='solomon')
    #display(solomon_data)
finally:
    # Clean up the temporary file
    os.remove(temp_file_path)


In [7]:
n = len(solomon_data['node_coord'])
m = solomon_data['vehicles']
q = solomon_data['capacity']
d = solomon_data['demand'].tolist()
a = solomon_data['time_window'][:, 0].tolist()
b = solomon_data['time_window'][:, 1].tolist()
s = solomon_data['service_time'].tolist()

print(f"Number of locations (n): {n}")
print(f"Number of vehicles (m): {m}")
print(f"Vehicle capacity (q): {q}")
print(f"Demand (d): {d}")
print(f"Ready time (a): {a}")
print(f"Due date (b): {b}")
print(f"Service time (s): {s}")

# Extract coordinates for distance calculation
coords = solomon_data['node_coord']

# Initialize the cost matrix
c = np.zeros((n, n))

# Calculate Euclidean distances
for i in range(n):
    for j in range(n):
        if i == j:
            c[i, j] = 0
        else:
            c[i, j] = math.dist(coords[i], coords[j])

# Convert to list of lists (if DIDPPY expects this format) and round to nearest integer as typically done in VRP
c = [[int(round(val)) for val in row] for row in c]

print(f"\nTravel cost matrix (c):\n{np.array(c)}")

Number of locations (n): 101
Number of vehicles (m): 25
Vehicle capacity (q): 200
Demand (d): [0, 10, 30, 10, 10, 10, 20, 20, 20, 10, 10, 10, 20, 30, 10, 40, 40, 20, 20, 10, 10, 20, 20, 10, 10, 40, 10, 10, 20, 10, 10, 20, 30, 40, 20, 10, 10, 20, 30, 20, 10, 10, 20, 10, 10, 10, 30, 10, 10, 10, 10, 10, 10, 20, 40, 10, 30, 40, 30, 10, 20, 10, 20, 50, 10, 10, 10, 10, 10, 10, 30, 20, 10, 10, 50, 20, 10, 10, 20, 10, 10, 30, 20, 10, 20, 30, 10, 20, 30, 10, 10, 10, 20, 40, 10, 30, 10, 30, 20, 10, 20]
Ready time (a): [0, 912, 825, 65, 727, 15, 621, 170, 255, 534, 357, 448, 652, 30, 567, 384, 475, 99, 179, 278, 10, 914, 812, 732, 65, 169, 622, 261, 546, 358, 449, 200, 31, 87, 751, 283, 665, 383, 479, 567, 264, 166, 68, 16, 359, 541, 448, 1054, 632, 1001, 815, 725, 912, 286, 186, 95, 385, 35, 471, 651, 562, 531, 262, 171, 632, 76, 826, 12, 734, 916, 387, 293, 450, 478, 353, 997, 203, 574, 109, 668, 769, 47, 369, 265, 458, 555, 173, 85, 645, 737, 20, 836, 368, 475, 285, 196, 95, 561, 30, 743, 647]

In [8]:
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_sol_file:
    temp_sol_file.write(C101_sol)
    temp_sol_file_path = temp_sol_file.name

try:
    solomon_solution = vrplib.read_solution(temp_sol_file_path)
    print("Solution Data:")
    display(solomon_solution)
finally:
    os.remove(temp_sol_file_path)

Solution Data:


{'routes': [[5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1, 75],
  [13, 17, 18, 19, 15, 16, 14, 12],
  [20, 24, 25, 27, 29, 30, 28, 26, 23, 22, 21],
  [32, 33, 31, 35, 37, 38, 39, 36, 34],
  [43, 42, 41, 40, 44, 46, 45, 48, 51, 50, 52, 49, 47],
  [57, 55, 54, 53, 56, 58, 60, 59],
  [67, 65, 63, 62, 74, 72, 61, 64, 68, 66, 69],
  [81, 78, 76, 71, 70, 73, 77, 79, 80],
  [90, 87, 86, 83, 82, 84, 85, 88, 89, 91],
  [98, 96, 95, 94, 92, 93, 97, 100, 99]],
 'cost': 827.3}

# **Approach 1: 1-transition model**

In [ ]:
# =====================================================================================
# 2. DIDP Model Definition
# =====================================================================================
model = dp.Model()

# Object types for customers/locations and vehicles
customer = model.add_object_type(number=n)
vehicle = model.add_object_type(number=m)

# -------------------- State Variables --------------------
# Set of unvisited customers
unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))

# Per-vehicle state variables, stored in Python lists for easy access
vehicle_locations = [
  model.add_element_var(object_type=customer, target=0, name=f"loc_v{v}")
  for v in range(m)
]
vehicle_loads = [
  model.add_int_var(target=0, name=f"load_v{v}")
  for v in range(m)
]
vehicle_times = [
  model.add_int_resource_var(target=0, less_is_better=True, name=f"time_v{v}")
  for v in range(m)
]

# -------------------- Tables of Constants --------------------
demand = model.add_int_table(d)
ready_time = model.add_int_table(a)
due_time = model.add_int_table(b)
service_time = model.add_int_table(s)
travel_time = model.add_int_table(c)

# -------------------- Transitions --------------------
# Transition to visit a customer j with a vehicle v
for v in range(m):
  for j in range(1, n):
      # Expression for arrival time at customer j with vehicle v
      arrival_time = dp.max(
          vehicle_times[v] + travel_time[vehicle_locations[v], j], ready_time[j]
      )

      # Expression for time vehicle v becomes free after serving j
      departure_time = arrival_time + service_time[j]

      visit_transition = dp.Transition(
          name=f"visit_{j}_with_vehicle_{v}",
          cost=travel_time[vehicle_locations[v], j] + dp.IntExpr.state_cost(),
          preconditions=[
              unvisited.contains(j),
              vehicle_loads[v] + demand[j] <= q,
              arrival_time <= due_time[j],
          ],
          effects=[
              (unvisited, unvisited.remove(j)),
              (vehicle_locations[v], j),
              (vehicle_loads[v], vehicle_loads[v] + demand[j]),
              (vehicle_times[v], departure_time),
          ],
      )
      model.add_transition(visit_transition)

# Transitions for each vehicle to return to the depot after all customers are served
for v in range(m):
  return_to_depot_transition = dp.Transition(
      name=f"return_vehicle_{v}_to_depot",
      cost=travel_time[vehicle_locations[v], 0] + dp.IntExpr.state_cost(),
      preconditions=[unvisited.is_empty(), vehicle_locations[v] != 0],
      effects=[(vehicle_locations[v], 0)],
  )
  model.add_transition(return_to_depot_transition)

# -------------------- Base Case --------------------
# All customers visited AND all vehicles are at the depot
base_conditions = [unvisited.is_empty()]
for v in range(m):
  base_conditions.append(vehicle_locations[v] == 0)
model.add_base_case(base_conditions)

# -------------------- State Constraints and Dual Bounds --------------------
# State constraint: if a customer is unvisited, it must be reachable in time by at least one vehicle
# This is complex to model perfectly, so we rely on transition preconditions for feasibility.
# A simpler constraint is that remaining capacity must meet remaining demand.
total_capacity_left = sum([(q - load) for load in vehicle_loads])
model.add_state_constr(total_capacity_left >= demand[unvisited])

# Dual bound: sum of minimum travel costs to remaining customers
min_to = model.add_int_table(
  [min(c[k][j] for k in range(n) if k != j) if j != 0 else 0 for j in range(n)]
)
model.add_dual_bound(min_to[unvisited])

# =====================================================================================
# 3. Solving the Model
# =====================================================================================
print("Solving CVRPTW model...")
# Using CABS solver, which is a good general-purpose choice
solver = dp.CABS(model, quiet=True)
solution = solver.search()

if solution.is_infeasible:
  print("The problem is infeasible.")
else:
  print("\nSolution Found:")
  print("Transitions to apply:")
  for t in solution.transitions:
      print(f"- {t.name}")
  print(f"\nOptimal Cost: {solution.cost}")
  print(f"Is optimal: {solution.is_optimal}")


Solving CVRPTW model...

Solution Found:
Transitions to apply:
- visit_1_with_vehicle_0
- visit_3_with_vehicle_0
- visit_2_with_vehicle_1
- return_vehicle_1_to_depot
- return_vehicle_0_to_depot

Optimal Cost: 20
Is optimal: True


# **Approach 2: 2-transition models**

In [ ]:
# =====================================================================================
# 2. DIDP Model Definition
# =====================================================================================
model = dp.Model()

# Object types for customers/locations and vehicles
customer = model.add_object_type(number=n)
vehicle = model.add_object_type(number=m)

# -------------------- State Variables --------------------
# Set of unvisited customers
unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))

# Per-vehicle state variables, stored in Python lists for easy access
vehicle_locations = [
  model.add_element_var(object_type=customer, target=0, name=f"loc_v{v}")
  for v in range(m)
]
vehicle_loads = [
  model.add_int_var(target=0, name=f"load_v{v}")
  for v in range(m)
]
vehicle_times = [
  model.add_int_resource_var(target=0, less_is_better=True, name=f"time_v{v}")
  for v in range(m)
]

chosen_customer = model.add_element_var(object_type=customer, target=0, name="chosen_customer")
alpha = model.add_int_var(target=0, name="alpha")

# -------------------- Tables of Constants --------------------
demand = model.add_int_table(d)
ready_time = model.add_int_table(a)
due_time = model.add_int_table(b)
service_time = model.add_int_table(s)
travel_time = model.add_int_table(c)

# -------------------- Transitions --------------------
# Choose customer j to be visited next
for j in range(1, n):
  choosing_customer_transition = dp.Transition(
      name=f"choose_customer_{j}_to_visit",
      cost=dp.IntExpr.state_cost(),
      preconditions =[
          unvisited.contains(j),
           alpha == 0
           ],
      effects=[
          (chosen_customer, j),
           (alpha, 1)
           ],
  )
  model.add_transition(choosing_customer_transition)

# Transition to visit a customer j with a vehicle v
for v in range(m):

    arrival_time = dp.max(
        vehicle_times[v] + travel_time[vehicle_locations[v], chosen_customer],
        ready_time[chosen_customer]
    )

    departure_time = arrival_time + service_time[chosen_customer]

    visit_transition = dp.Transition(
        name=f"visit_chosen_customer_with_vehicle_{v}",
        cost=travel_time[vehicle_locations[v], chosen_customer] + dp.IntExpr.state_cost(),
        preconditions=[
            unvisited.contains(chosen_customer),
            vehicle_loads[v] + demand[chosen_customer] <= q,
            arrival_time <= due_time[chosen_customer],
            alpha == 1,
        ],
        effects=[
            (unvisited, unvisited.remove(chosen_customer)),
            (vehicle_locations[v], chosen_customer),
            (vehicle_loads[v], vehicle_loads[v] + demand[chosen_customer]),
            (vehicle_times[v], departure_time),
            (alpha, 0),
        ],
    )

    model.add_transition(visit_transition)


# Transitions for each vehicle to return to the depot after all customers are served
for v in range(m):
  return_to_depot_transition = dp.Transition(
      name=f"return_vehicle_{v}_to_depot",
      cost=travel_time[vehicle_locations[v], 0] + dp.IntExpr.state_cost(),
      preconditions=[unvisited.is_empty(), vehicle_locations[v] != 0],
      effects=[(vehicle_locations[v], 0)],
  )
  model.add_transition(return_to_depot_transition)

# -------------------- Base Case --------------------
# All customers visited AND all vehicles are at the depot
base_conditions = [unvisited.is_empty()]
for v in range(m):
  base_conditions.append(vehicle_locations[v] == 0)
model.add_base_case(base_conditions)

# -------------------- State Constraints and Dual Bounds --------------------
# State constraint: if a customer is unvisited, it must be reachable in time by at least one vehicle
# This is complex to model perfectly, so we rely on transition preconditions for feasibility.
# A simpler constraint is that remaining capacity must meet remaining demand.
total_capacity_left = sum([(q - load) for load in vehicle_loads])
model.add_state_constr(total_capacity_left >= demand[unvisited])

# Dual bound: sum of minimum travel costs to remaining customers
min_to = model.add_int_table(
  [min(c[k][j] for k in range(n) if k != j) if j != 0 else 0 for j in range(n)]
)
model.add_dual_bound(min_to[unvisited])

# =====================================================================================
# 3. Solving the Model
# =====================================================================================
print("Solving CVRPTW model...")
# Using CABS solver, which is a good general-purpose choice
solver = dp.CABS(model, quiet=True)# time_limit = 10)
solution = solver.search()

if solution.is_infeasible:
  print("The problem is infeasible.")
else:
  print("\nSolution Found:")
  print("Transitions to apply:")
  for t in solution.transitions:
      print(f"- {t.name}")
  print(f"\nOptimal Cost: {solution.cost}")
  print(f"Is optimal: {solution.is_optimal}")
